In [ ]:
using LinearAlgebra
using Statistics
using StaticArrays
using Random
using PythonPlot

## Sections 1 & 2: Safe Mode Spin Design and Orbit–Attitude Dynamics

In [ ]:
include("sources/safe_mode_gyrostat.jl")
include("sources/hw2_simulation.jl")

Random.seed!(12345)

I_body = @SMatrix [
    0.249  0.0    0.0
    0.0    0.187  0.02
    0.0    0.02   0.148
]

e_panel   = @SVector [0.0, 1.0, 0.0]   # solar panel normal (+b1 / body y-axis)
omega_des = rpm2rad(10.0)               # 10 RPM -> rad/s
w_des     = omega_des .* e_panel

println("omega_des = $(round(omega_des, digits=4)) rad/s")

In [ ]:
I_pert = perturb_inertia(I_body; sig_eig=0.03, sig_rot=deg2rad(2.0))
H_r, I_rotor_eff = required_rotor_momentum(I_pert, e_panel; ratio=1.2, omega=omega_des)

println("H_r = $(round(H_r, digits=4)) N·m·s,  I_rotor = $(round(I_rotor_eff, digits=4)) kg·m²")

In [ ]:
q0_att      = @SVector [1.0, 0.0, 0.0, 0.0]
delta_omega = 0.01 * omega_des

xi  = randn(3); xi -= dot(xi, e_panel)*e_panel; xi /= norm(xi)
w0_att = w_des + delta_omega .* xi

sol_att = integrate_gyrostat!(I_pert, w0_att, q0_att, H_r, e_panel;
    tspan=(0.0, 800.0), reltol=1e-9, abstol=1e-12)

t_att = sol_att.t
w1 = [sol_att.u[k][5] for k in 1:length(t_att)]
w2 = [sol_att.u[k][6] for k in 1:length(t_att)]
w3 = [sol_att.u[k][7] for k in 1:length(t_att)]

fig_att, ax_att = subplots(3, 1, figsize=(8, 6), sharex=true)
ax_att[0].plot(t_att, w1, color="steelblue")
ax_att[0].set_ylabel("ω₁ (rad/s)"); ax_att[0].set_title("Safe-mode gyrostat: angular velocity components"); ax_att[0].grid(true, alpha=0.3)
ax_att[1].plot(t_att, w2, color="tomato")
ax_att[1].set_ylabel("ω₂ (rad/s)"); ax_att[1].grid(true, alpha=0.3)
ax_att[2].plot(t_att, w3, color="green")
ax_att[2].set_ylabel("ω₃ (rad/s)"); ax_att[2].set_xlabel("Time (s)"); ax_att[2].grid(true, alpha=0.3)
fig_att.tight_layout()
savefig("figs/safe_mode_angular_velocity.png"; dpi=150, bbox_inches="tight")

In [ ]:
alt_sso  = 480.0
RE_km    = 6378.1363
a_sso    = RE_km + alt_sso
i_sso    = compute_sso_inclination(alt_sso)
r0, v0   = coe_to_eci(a_sso, 0.0, i_sso, 0.0, 0.0, 0.0)

sun_vec  = @SVector [1.0, 0.0, 0.0]
q0_orbit = rotation_quat_between(e_panel, sun_vec)

xi2  = randn(3); xi2 -= dot(xi2, e_panel)*e_panel; xi2 /= norm(xi2)
w0_orbit = w_des + delta_omega .* xi2

x0 = Vector{Float64}(undef, 13)
x0[1:3] = r0; x0[4:6] = v0; x0[7:10] = q0_orbit; x0[11:13] = w0_orbit

sol_orb = integrate_orbit_gyrostat!(x0, I_pert, H_r, e_panel;
    tspan=(0.0, 2000.0), drag=false, reltol=1e-9, abstol=1e-12)

t_orb     = sol_orb.t
theta_err = pointing_error(sol_orb, e_panel, sun_vec)

fig_orb, ax_orb = subplots(2, 1, figsize=(8, 6), sharex=true)
ax_orb[0].plot(t_orb, [u[7]  for u in sol_orb.u], label="q0", color="black")
ax_orb[0].plot(t_orb, [u[8]  for u in sol_orb.u], label="q1", color="steelblue")
ax_orb[0].plot(t_orb, [u[9]  for u in sol_orb.u], label="q2", color="tomato")
ax_orb[0].plot(t_orb, [u[10] for u in sol_orb.u], label="q3", color="green")
ax_orb[0].set_ylabel("Quaternion components"); ax_orb[0].set_title("Orbit–gyrostat: quaternion and pointing error")
ax_orb[0].legend(loc="upper right"); ax_orb[0].grid(true, alpha=0.3)
ax_orb[1].plot(t_orb, theta_err, color="purple")
ax_orb[1].set_xlabel("Time (s)"); ax_orb[1].set_ylabel("Pointing error (deg)"); ax_orb[1].grid(true, alpha=0.3)
fig_orb.tight_layout()
savefig("figs/orbit_attitude_and_pointing.png"; dpi=150, bbox_inches="tight")

## Section 3: Attitude Sensors — Noise Models and Monte Carlo Verification

In [2]:
# q = [scalar; vector] = [q0; q1; q2; q3]
# Q(q) maps body -> inertial: r_N = Q(q) * r_B

function hat(v)
    return [0 -v[3] v[2];
            v[3] 0 -v[1];
            -v[2] v[1] 0]
end

function unhat(S)
    return 0.5*[S[3,2]-S[2,3];
                S[1,3]-S[3,1];
                S[2,1]-S[1,2]]
end

H = [zeros(1,3); I(3)]

function L(q)
    return [q[1]          -q[2:4]';
            q[2:4]    q[1]*I(3) + hat(q[2:4])]
end

function R(q)
    return [q[1]          -q[2:4]';
            q[2:4]    q[1]*I(3) - hat(q[2:4])]
end

function G(q)
    return L(q)*H
end

function Q(q)
    return H'*(R(q)'*L(q))*H
end

function expq(phi)
    theta = norm(phi)
    return [cos(theta); phi*sinc(theta/pi)]
end

function logq(q)
    c = q[1]
    s = norm(q[2:4])
    theta = atan(s, c)
    return q[2:4] / sinc(theta/pi)
end

logq (generic function with 1 method)

In [3]:
# Sensor covariance matrices built from 1-sigma values

# Sun sensor: AAC Clyde Space SS200, 0.3 deg (1-sigma)
sigma_sun = 0.3 * (pi/180)
R_sun = sigma_sun^2 * Matrix(I, 3, 3)

# Magnetometer: NewSpace Systems, 1.0 deg effective directional uncertainty
sigma_mag = 1.0 * (pi/180)
R_mag = sigma_mag^2 * Matrix(I, 3, 3)

# Star tracker: Blue Canyon NST, 6 arcsec cross-boresight, 40 arcsec about-boresight
sigma_st_cross = 6.0  * (pi/180) / 3600
sigma_st_bore  = 40.0 * (pi/180) / 3600
R_st = Diagonal([sigma_st_cross^2, sigma_st_cross^2, sigma_st_bore^2])

println("sigma_sun = $(round(sigma_sun*(180/pi), digits=3)) deg")
println("sigma_mag = $(round(sigma_mag*(180/pi), digits=3)) deg")
println("sigma_st  = $(round(sigma_st_cross*(180/pi)*3600, digits=1)) / $(round(sigma_st_bore*(180/pi)*3600, digits=1)) arcsec (cross/bore)")

sigma_sun = 0.3 deg
sigma_mag = 1.0 deg
sigma_st  = 6.0 / 40.0 arcsec (cross/bore)


In [4]:
# Sample noise from N(0, R_cov)
function sample_noise(R_cov::AbstractMatrix)
    L_chol = cholesky(Symmetric(Matrix(R_cov))).L
    return L_chol * randn(3)
end

# Vector measurement: normalize(v_true + noise)
function noisy_vector_meas(v_true::AbstractVector, R_cov::AbstractMatrix)
    v = v_true + sample_noise(R_cov)
    return v / norm(v)
end

# Star tracker: perturb true quaternion by a small rotation delta_phi ~ N(0, R_cov)
function noisy_star_tracker_meas(q_true::AbstractVector, R_cov::AbstractMatrix)
    delta_phi = sample_noise(R_cov)
    delta_q = [1.0; 0.5 * delta_phi]
    delta_q /= norm(delta_q)
    return normalize(L(q_true) * delta_q)
end

noisy_star_tracker_meas (generic function with 1 method)

In [ ]:
N_mc = 10_000

# Sun sensor
v_sun_true = normalize([1.0, 0.0, 0.0])
errors_sun = zeros(N_mc)
for i = 1:N_mc
    v_meas = noisy_vector_meas(v_sun_true, R_sun)
    errors_sun[i] = acos(clamp(dot(v_meas, v_sun_true), -1.0, 1.0)) * (180/pi)
end

sigma_sun_deg = sigma_sun * (180/pi)
println("Sun sensor (N=$N_mc): mean=$(round(mean(errors_sun),digits=4)) deg  (theory=$(round(sigma_sun_deg*sqrt(pi/2),digits=4)) deg),  RMS=$(round(sqrt(mean(errors_sun.^2)),digits=4)) deg")

# Magnetometer
v_mag_true = normalize([0.0, 1.0, 0.0])
errors_mag = zeros(N_mc)
for i = 1:N_mc
    v_meas = noisy_vector_meas(v_mag_true, R_mag)
    errors_mag[i] = acos(clamp(dot(v_meas, v_mag_true), -1.0, 1.0)) * (180/pi)
end

sigma_mag_deg = sigma_mag * (180/pi)
println("Magnetometer (N=$N_mc): mean=$(round(mean(errors_mag),digits=4)) deg (theory=$(round(sigma_mag_deg*sqrt(pi/2),digits=4)) deg),  RMS=$(round(sqrt(mean(errors_mag.^2)),digits=4)) deg")

# Star tracker — recover delta_phi from each noisy measurement
q_st_true = [1.0, 0.0, 0.0, 0.0]
delta_phi_samples = zeros(3, N_mc)
for i = 1:N_mc
    q_meas = noisy_star_tracker_meas(q_st_true, Matrix(R_st))
    delta_phi_samples[:, i] = 2 * logq(q_meas)
end

sample_cov_st = (delta_phi_samples * delta_phi_samples') / (N_mc - 1)
s1 = round(sqrt(sample_cov_st[1,1])*(180/pi)*3600, digits=2)
s2 = round(sqrt(sample_cov_st[2,2])*(180/pi)*3600, digits=2)
s3 = round(sqrt(sample_cov_st[3,3])*(180/pi)*3600, digits=2)
println("Star tracker (N=$N_mc): sigma = $s1, $s2, $s3 arcsec (intended: $(round(sigma_st_cross*(180/pi)*3600,digits=1)), $(round(sigma_st_cross*(180/pi)*3600,digits=1)), $(round(sigma_st_bore*(180/pi)*3600,digits=1)) arcsec)")

In [ ]:
fig, (ax1, ax2, ax3) = subplots(1, 3, figsize=(13, 4))

# Sun sensor: angular error histogram vs Rayleigh PDF
theta_vals = range(0.0, maximum(errors_sun)*1.15, length=300)
rayleigh_sun = @. (theta_vals*(pi/180)) / sigma_sun^2 *
    exp(-(theta_vals*(pi/180))^2 / (2*sigma_sun^2)) * (pi/180)

ax1.hist(errors_sun; bins=55, density=true, color="steelblue", alpha=0.7,
    label="Samples (N=$N_mc)")
ax1.plot(collect(theta_vals), rayleigh_sun; color="tomato", linewidth=2.0,
    label="Rayleigh(sigma=$(round(sigma_sun_deg, digits=3)) deg)")
ax1.axvline(sigma_sun_deg; color="gray", linewidth=1.0, linestyle="--",
    label="1-sigma = $(round(sigma_sun_deg, digits=3)) deg")
ax1.set_xlabel("Angular error (deg)")
ax1.set_ylabel("Probability density")
ax1.set_title("Sun Sensor Noise Verification")
ax1.legend(fontsize=7)
ax1.grid(true, alpha=0.3)

# Magnetometer: angular error histogram vs Rayleigh PDF
theta_vals_m = range(0.0, maximum(errors_mag)*1.15, length=300)
rayleigh_mag = @. (theta_vals_m*(pi/180)) / sigma_mag^2 *
    exp(-(theta_vals_m*(pi/180))^2 / (2*sigma_mag^2)) * (pi/180)

ax2.hist(errors_mag; bins=55, density=true, color="steelblue", alpha=0.7,
    label="Samples (N=$N_mc)")
ax2.plot(collect(theta_vals_m), rayleigh_mag; color="tomato", linewidth=2.0,
    label="Rayleigh(sigma=$(round(sigma_mag_deg, digits=3)) deg)")
ax2.axvline(sigma_mag_deg; color="gray", linewidth=1.0, linestyle="--",
    label="1-sigma = $(round(sigma_mag_deg, digits=3)) deg")
ax2.set_xlabel("Angular error (deg)")
ax2.set_title("Magnetometer Noise Verification")
ax2.legend(fontsize=7)
ax2.grid(true, alpha=0.3)

# Star tracker: per-axis error histograms in arcsec
dp_arcsec = delta_phi_samples * (180/pi) * 3600
ax3.hist(dp_arcsec[1,:]; bins=55, density=true, color="steelblue", alpha=0.5,
    label="Axis 1 cross ($(round(sigma_st_cross*(180/pi)*3600,digits=1)) arcsec)")
ax3.hist(dp_arcsec[2,:]; bins=55, density=true, color="tomato", alpha=0.5,
    label="Axis 2 cross")
ax3.hist(dp_arcsec[3,:]; bins=55, density=true, color="green", alpha=0.5,
    label="Axis 3 bore ($(round(sigma_st_bore*(180/pi)*3600,digits=1)) arcsec)")
ax3.set_xlabel("Attitude perturbation delta_phi (arcsec)")
ax3.set_title("Star Tracker Noise Verification")
ax3.legend(fontsize=7)
ax3.grid(true, alpha=0.3)

tight_layout()
savefig("figs/sensor_noise_verification.png"; dpi=150, bbox_inches="tight")
println("Saved figs/sensor_noise_verification.png")

## Section 4: Static Attitude Estimation — Wahba's Problem

In [ ]:
Random.seed!(7)

q_true = expq(0.5 * randn(3))
q_true = q_true / norm(q_true)
Q_true = Q(q_true)

# Inertial reference vectors (Sun along ECI +x, magnetic field direction arbitrary but fixed)
b_sun_N = [1.0, 0.0, 0.0]
b_mag_N = normalize([0.3, 0.5, 0.8])

# True body-frame vectors: r_B = Q_true' * r_N
b_sun_B_true = Q_true' * b_sun_N
b_mag_B_true = Q_true' * b_mag_N

# Inverse-variance weights, normalized to sum to 1
w_sun = (1/sigma_sun^2) / (1/sigma_sun^2 + 1/sigma_mag^2)
w_mag = 1 - w_sun

println("w_sun = $(round(w_sun, digits=4)),  w_mag = $(round(w_mag, digits=4))")

In [ ]:
# SVD Wahba solver (Markley 1988):
# B = sum w_k * r_B_k * r_N_k',  then Q_est = V * diag(1,1,det(V)*det(U)) * U'
function wahba_svd(r_B_list, r_N_list, weights)
    B = zeros(3, 3)
    for k in eachindex(r_B_list)
        B += weights[k] * r_B_list[k] * r_N_list[k]'
    end
    F = svd(B)
    Q_est = F.V * Diagonal([1.0, 1.0, det(F.V)*det(F.U)]) * F.U'
    return Q_est
end

Q_svd_clean = wahba_svd([b_sun_B_true, b_mag_B_true], [b_sun_N, b_mag_N], [w_sun, w_mag])
println("SVD error noiseless: $(round((180/pi)*norm(unhat(log(Q_svd_clean'*Q_true))), sigdigits=3)) deg")

In [ ]:
# Davenport q-method: largest eigenvector of D = sum w_k * L(H*r_N_k)' * R(H*r_B_k)
function wahba_qmethod(r_B_list, r_N_list, weights)
    D = zeros(4, 4)
    for k in eachindex(r_B_list)
        D += weights[k] * L(H * r_N_list[k])' * R(H * r_B_list[k])
    end
    F_eig = eigen(D)
    q_est = F_eig.vectors[:, argmax(F_eig.values)]
    return q_est / norm(q_est)
end

q_qm_clean = wahba_qmethod([b_sun_B_true, b_mag_B_true], [b_sun_N, b_mag_N], [w_sun, w_mag])
println("q-method noiseless error: $(round((180/pi)*norm(unhat(log(Q(q_qm_clean)'*Q_true))), sigdigits=3)) deg")

In [ ]:
b_sun_B_meas = noisy_vector_meas(b_sun_B_true, R_sun)
b_mag_B_meas = noisy_vector_meas(b_mag_B_true, R_mag)

sun_noise_deg = acos(clamp(dot(b_sun_B_meas, b_sun_B_true), -1.0, 1.0)) * (180/pi)
mag_noise_deg = acos(clamp(dot(b_mag_B_meas, b_mag_B_true), -1.0, 1.0)) * (180/pi)

Q_svd = wahba_svd([b_sun_B_meas, b_mag_B_meas], [b_sun_N, b_mag_N], [w_sun, w_mag])
err_svd_trial = (180/pi) * norm(unhat(log(Q_svd' * Q_true)))

q_qm = wahba_qmethod([b_sun_B_meas, b_mag_B_meas], [b_sun_N, b_mag_N], [w_sun, w_mag])
err_qm_trial = (180/pi) * norm(unhat(log(Q(q_qm)' * Q_true)))

println("Single noisy trial:")
println("  sun noise = $(round(sun_noise_deg, digits=4)) deg,  mag noise = $(round(mag_noise_deg, digits=4)) deg")
println("  SVD error = $(round(err_svd_trial, digits=4)) deg,  q-method error = $(round(err_qm_trial, digits=4)) deg")

In [ ]:
Random.seed!(42)
N_wahba = 5_000

errors_svd     = zeros(N_wahba)
errors_qmethod = zeros(N_wahba)

for i = 1:N_wahba
    r_B_meas = [noisy_vector_meas(b_sun_B_true, R_sun), noisy_vector_meas(b_mag_B_true, R_mag)]
    errors_svd[i]     = (180/pi) * norm(unhat(log(wahba_svd(r_B_meas, [b_sun_N, b_mag_N], [w_sun, w_mag])' * Q_true)))
    errors_qmethod[i] = (180/pi) * norm(unhat(log(Q(wahba_qmethod(r_B_meas, [b_sun_N, b_mag_N], [w_sun, w_mag]))' * Q_true)))
end

println("Monte Carlo (N=$N_wahba):")
println("  SVD:      mean=$(round(mean(errors_svd),digits=4)) deg,  RMS=$(round(sqrt(mean(errors_svd.^2)),digits=4)) deg,  95th=$(round(quantile(errors_svd,0.95),digits=4)) deg")
println("  q-method: mean=$(round(mean(errors_qmethod),digits=4)) deg,  RMS=$(round(sqrt(mean(errors_qmethod.^2)),digits=4)) deg,  95th=$(round(quantile(errors_qmethod,0.95),digits=4)) deg")

In [5]:
fig2, (ax_hist, ax_cdf) = subplots(1, 2, figsize=(11, 4))

max_err = max(maximum(errors_svd), maximum(errors_qmethod))
bins_w = range(0.0, max_err * 1.05, length=60)

ax_hist.hist(errors_svd;
    bins=collect(bins_w), density=true, alpha=0.6,
    color="steelblue", label="SVD")
ax_hist.hist(errors_qmethod;
    bins=collect(bins_w), density=true, alpha=0.6,
    color="tomato", label="Davenport q-method")
ax_hist.axvline(mean(errors_svd);
    color="steelblue", linewidth=1.5, linestyle="--",
    label="SVD mean = $(round(mean(errors_svd), digits=3)) deg")
ax_hist.axvline(mean(errors_qmethod);
    color="tomato", linewidth=1.5, linestyle="--",
    label="q-method mean = $(round(mean(errors_qmethod), digits=3)) deg")
ax_hist.set_xlabel("3D attitude error (deg)")
ax_hist.set_ylabel("Probability density")
ax_hist.set_title("Wahba Error Distribution (N=$N_wahba)")
ax_hist.legend(fontsize=8)
ax_hist.grid(true, alpha=0.3)

p_vals = range(0.0, 1.0, length=N_wahba)
ax_cdf.plot(sort(errors_svd),     collect(p_vals); color="steelblue", linewidth=1.5, label="SVD")
ax_cdf.plot(sort(errors_qmethod), collect(p_vals); color="tomato",    linewidth=1.5, label="Davenport q-method")
ax_cdf.axhline(0.68; color="gray", linewidth=0.8, linestyle=":", label="68th percentile")
ax_cdf.axhline(0.95; color="gray", linewidth=0.8, linestyle="--", label="95th percentile")
ax_cdf.set_xlabel("3D attitude error (deg)")
ax_cdf.set_ylabel("Cumulative probability")
ax_cdf.set_title("Empirical CDF")
ax_cdf.legend(fontsize=8)
ax_cdf.grid(true, alpha=0.3)

tight_layout()
savefig("figs/wahba_comparison.png"; dpi=150, bbox_inches="tight")
println("Saved figs/wahba_comparison.png")

LoadError: UndefVarError: `errors_svd` not defined in `Main`
Suggestion: check for spelling errors or missing imports.